# For playing around with plotting and data parameters in analyze_friction.py

In [1]:
import argparse
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.signal import savgol_filter
from scipy.interpolate import interp1d
import yaml

In [5]:
# ---------------------------------------------------------------------------
# Parse CLI arguments
# ---------------------------------------------------------------------------
_SCRIPT_DIR = "/Users/jazlabamini/Documents/nikashap/orcaSDK_tutorials/tests/"

EXP_DATE = "26_04_21-14_25_29"

# ---------------------------------------------------------------------------
# Resolve experiment directory and load the saved YAML snapshot
# ---------------------------------------------------------------------------
# Base data dir is relative to script; the date subfolder identifies the experiment
_BASE_DATA_DIR = os.path.normpath(os.path.join(_SCRIPT_DIR, "../data"))
EXPERIMENT_DIR = os.path.join(_BASE_DATA_DIR, EXP_DATE)

_PARAMS_PATH = os.path.join(EXPERIMENT_DIR, "calibration_params.yaml")
if not os.path.exists(_PARAMS_PATH):
    print(f"ERROR: No calibration_params.yaml found in {EXPERIMENT_DIR}")
    print("Make sure you collected data with collect_calibration_data.py first.")
    sys.exit(1)

with open(_PARAMS_PATH, "r") as f:
    PARAMS = yaml.safe_load(f)

M_SHAFT_KG = PARAMS["mass_shaft_kg"]
G_MMPSS = 9810.0  # gravitational acceleration in mm/s²
WEIGHT_MN = M_SHAFT_KG * G_MMPSS  # normal force in mN

VELOCITY_ESTIMATOR = PARAMS.get("velocity_estimator", "SG").upper()
SAVGOL_WINDOW = PARAMS["savgol_window"]
SAVGOL_ORDER = PARAMS["savgol_order"]
RW_WINDOW = PARAMS.get("rw_window", 5)
T_IGNORE_S = PARAMS["t_ignore_s"]
F_STATIC_LOW_MN = PARAMS["static_friction_low_mN"]
F_STATIC_HIGH_MN = PARAMS["static_friction_high_mN"]
STREAM_FORCE_SHIFT = PARAMS.get("stream_force_shift", 0)
USB_CHIP = PARAMS.get("usb_chip", "unknown")

## Functions

In [7]:
def load_master_data(filepath):
    """Load a master .npz file and split into per-trial dicts.

    Handles both constant-force data (calibration_dynamic_friction.npz) and
    ramp-down data (calibration_rampdown_friction.npz).  In both cases,
    force_commanded_mN is returned as a per-sample array.  A scalar
    force_label_mN is also included for grouping / plotting:
      - constant-force trials: the commanded force level
      - ramp-down trials: the target (post-switch) force
    """
    data = np.load(filepath, allow_pickle=True)

    n_trials = int(data["n_trials"])
    boundaries = data["trial_boundaries"]

    # Determine procedure type: explicit tag in new data, fallback for old data
    if "procedure_type" in data:
        procedure_type = str(data["procedure_type"])
    else:
        procedure_type = "constant"

    is_rampdown = (procedure_type == "rampdown")

    t_stream_all = data["t_stream"]
    position_um_all = data["position_um"]
    force_mN_all = data["force_mN"]
    t_accel_all = data["t_accel"]
    accel_mmpss_all = data["accel_mmpss"]

    if is_rampdown:
        force_cmd_all = data["force_commanded_mN"]
        trial_labels = data["trial_target_force_mN"]
        trial_switch_indices = data["trial_switch_index"]
    else:
        force_cmd_all = (data["force_commanded_mN"] if "force_commanded_mN" in data
                         else None)
        force_cmd_per_trial = data["trial_force_commanded_mN"]

    trials = {}
    for i in range(n_trials):
        lo, hi = boundaries[i], boundaries[i + 1]

        if is_rampdown:
            switch_idx = int(trial_switch_indices[i])
            if switch_idx < 0:
                continue
            # Deleted filtering out initial push from here
            force_cmd = force_cmd_all[lo:hi].astype(np.float64)
            label = int(trial_labels[i])
        elif force_cmd_all is not None:
            force_cmd = force_cmd_all[lo:hi].astype(np.float64)
            label = int(force_cmd_per_trial[i])
        else:
            force_cmd = np.full(hi - lo, force_cmd_per_trial[i], dtype=np.float64)
            label = int(force_cmd_per_trial[i])

        t_stream = t_stream_all[lo:hi]
        if len(t_stream) == 0:
            continue
        t0 = t_stream[0]

        trials[i + 1] = {
            "trial_num": i + 1,
            "force_commanded_mN": force_cmd,
            "force_label_mN": label,
            "t_stream_s": t_stream - t0,
            "t_accel_s": t_accel_all[lo:hi] - t0,
            "position_um": position_um_all[lo:hi].astype(np.float64),
            "force_mN": force_mN_all[lo:hi].astype(np.float64),
            "accel_mmpss": accel_mmpss_all[lo:hi].astype(np.float64),
        }

    if is_rampdown:
        force_levels = sorted(set(int(trial_labels[i]) for i in range(n_trials)))
    else:
        force_levels = data["force_levels_mN"].tolist()

    metadata = {
        "mass_shaft_kg": float(data["mass_shaft_kg"]),
        "motor_min_um": int(data["motor_min_um"]),
        "motor_max_um": int(data["motor_max_um"]),
        "force_levels_mN": force_levels,
    }

    return trials, metadata

In [10]:
def _velocity_savgol(pos_um, t, savgol_window, savgol_order):
    """Velocity via Savitzky-Golay analytical derivative (deriv=1).
    Using delta=mean(dt) because savgol_filter assumes uniform spacing.
    """
    mean_dt = np.mean(np.diff(t))
    velocity_um_s = savgol_filter(pos_um, window_length=savgol_window,
                                  polyorder=savgol_order, deriv=1, delta=mean_dt)
    return velocity_um_s / 1000.0


def _velocity_rolling_window(pos_um, t, rw_window):
    """Velocity via rolling mean of finite differences.

    v[i] is the mean of the last ``rw_window`` finite-difference
    velocity samples: (pos[i]-pos[i-1])/dt, ..., (pos[i-k+1]-pos[i-k])/dt.
    The first ``rw_window`` samples use a growing window (all differences
    available so far) so no samples are NaN.
    """
    dt = np.diff(t)                       # (n-1,)
    v_inst = np.diff(pos_um) / dt / 1000.0  # instantaneous velocity in mm/s, (n-1,)

    n = len(pos_um)
    velocity_mm_s = np.zeros(n)
    # v_inst[i] = velocity between sample i and i+1.
    # We assign v_inst[i] as the velocity "at" sample i+1 (backward difference).
    # Sample 0 gets the first available difference.
    cumsum = np.cumsum(np.concatenate([[0.0], v_inst]))
    for i in range(n):
        if i == 0:
            velocity_mm_s[0] = v_inst[0]  # only one difference available
        else:
            lo = max(0, i - rw_window)
            velocity_mm_s[i] = (cumsum[i] - cumsum[lo]) / (i - lo)

    return velocity_mm_s


def compute_friction_for_trial(trial, velocity_estimator="SG",
                               savgol_window=15, savgol_order=2,
                               rw_window=5, shift=0, shift_kinematics=False):
    """
    For one trial, compute:
      - velocity (via the chosen estimator: SG or RW)
      - F_friction = F_sensed_shifted - m * a
      - mu_d = F_friction / (m * g) (this is signed, NOT absolute value)

    Parameters
    ----------
    velocity_estimator : str
        "SG" for Savitzky-Golay (non-causal, offline), "RW" for rolling window (causal, online).
    savgol_window, savgol_order : int
        Parameters for the SG estimator (ignored when velocity_estimator=="RW").
    rw_window : int
        Number of finite-difference samples to average for the RW estimator
        (ignored when velocity_estimator=="SG").
    shift : int
        Frame shift to align force_sensed with acceleration. The motor stream
        (position, force_sensed) lags behind the real-time acceleration read
        by ``shift`` frames. So force_sensed[i+shift] corresponds to accel[i].
        We pair force_sensed[shift:n] with accel[0:n-shift].

    After shifting, only the overlapping region is kept — all arrays are
    trimmed to the same valid range.

    Returns a dict with all original fields (trimmed) plus velocity, friction,
    mu_d, force_commanded_mN (per-sample, trimmed), and
    force_sensed_unshifted_mN (sensed force at the same timepoint as
    position/velocity — NOT shift-aligned to accel).

    TODO: implement `shift_kinematics` parameter so that if True, then position and velocity data are also shifted by frame_shift
    """
    t = trial["t_stream_s"]
    pos_um = trial["position_um"]
    sensed_force = trial["force_mN"]
    force_commanded = trial["force_commanded_mN"]
    t_accel = trial["t_accel_s"]
    accel_raw = trial["accel_mmpss"]

    # Interpolate acceleration onto stream timestamps
    accel_interp_fn = interp1d(t_accel, accel_raw, kind="linear",
                               bounds_error=False, fill_value="extrapolate")
    accel_at_stream = accel_interp_fn(t)

    # Velocity estimation
    if velocity_estimator == "SG":
        velocity_mm_s = _velocity_savgol(pos_um, t, savgol_window, savgol_order)
    elif velocity_estimator == "RW":
        velocity_mm_s = _velocity_rolling_window(pos_um, t, rw_window)
    else:
        raise ValueError(f"Unknown velocity_estimator: {velocity_estimator!r}. "
                         f"Expected 'SG' or 'RW'.")

    n = len(t)

    # Apply frame shift to align force_sensed with acceleration.
    # Position, velocity, and force_commanded are NOT shifted — they stay at
    # the stream timepoint.  force_sensed_unshifted is also at the stream
    # timepoint (same frame as position).  force_aligned (used for friction
    # calculation) is the shifted force_sensed that corresponds to accel.
    if shift > 0:
        force_aligned = sensed_force[shift:]
        accel_aligned = accel_at_stream[:n - shift]
        t_trimmed = t[:n - shift]
        if shift_kinematics:
            print("\n---NOTE---\nPosition and Velocity are shifted according to frame_shift")
            pos_trimmed = pos_um[shift:]
            vel_trimmed = velocity_mm_s[shift:]
        else:
            pos_trimmed = pos_um[:n - shift]
            vel_trimmed = velocity_mm_s[:n - shift]
        force_cmd_trimmed = force_commanded[:n - shift]
        force_sensed_unshifted = sensed_force[:n - shift]
    elif shift < 0:
        abs_shift = abs(shift)
        force_aligned = sensed_force[:n - abs_shift]
        accel_aligned = accel_at_stream[abs_shift:]
        t_trimmed = t[abs_shift:]
        if shift_kinematics:
            print("\n---NOTE---\nPosition and Velocity are shifted according to frame_shift")
            pos_trimmed = pos_um[:n - abs_shift]
            vel_trimmed = velocity_mm_s[:n - abs_shift]
        else:
            pos_trimmed = pos_um[abs_shift:]
            vel_trimmed = velocity_mm_s[abs_shift:]
        force_cmd_trimmed = force_commanded[abs_shift:]
        force_sensed_unshifted = sensed_force[abs_shift:]
    else:
        force_aligned = sensed_force
        accel_aligned = accel_at_stream
        t_trimmed = t
        pos_trimmed = pos_um
        vel_trimmed = velocity_mm_s
        force_cmd_trimmed = force_commanded
        force_sensed_unshifted = sensed_force

    # F_friction = F_sensed_aligned - m * a  (all in mN, since kg * mm/s² = mN)
    f_friction_mN = force_aligned - M_SHAFT_KG * accel_aligned

    # Coefficient of dynamic friction
    mu_d = f_friction_mN / WEIGHT_MN

    return {
        "trial_num": trial["trial_num"],
        "force_label_mN": trial["force_label_mN"],
        "force_commanded_mN": force_cmd_trimmed,
        "t_stream_s": t_trimmed,
        "position_um": pos_trimmed,
        "force_mN": force_aligned,
        "force_sensed_unshifted_mN": force_sensed_unshifted,
        "accel_mmpss": accel_aligned,
        "accel_interp_mmpss": accel_aligned,
        "velocity_mm_s": vel_trimmed,
        "f_friction_mN": f_friction_mN,
        "mu_d": mu_d,
    }

## Plotting functions

In [ ]:
def plot_position_over_time(friction_trials, force_levels):
    """Plot position vs time for each force level (one subplot per level)"""
    pass

In [9]:
def plot_velocity_over_time(friction_trials, force_levels):
    """Plot velocity vs time for each force level (one subplot per level)."""
    fig, axes = plt.subplots(len(force_levels), 1,
                             figsize=(12, 4 * len(force_levels)), sharex=False)
    if len(force_levels) == 1:
        axes = [axes]

    colors = plt.cm.tab10(np.linspace(0, 1, 10))

    for i, force_mN in enumerate(force_levels):
        ax = axes[i]
        recs = [r for r in friction_trials if r["force_label_mN"] == force_mN]
        for j, r in enumerate(recs):
            ax.plot(r["t_stream_s"], r["velocity_mm_s"],
                    linewidth=0.8, color=colors[j % len(colors)],
                    label=f"Trial {r['trial_num']}")
        ax.set_ylabel("Velocity (mm/s)")
        ax.set_title(f"F_commanded = {force_mN} mN")
        ax.legend(fontsize=7, loc="upper right")
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time (s)")
    if VELOCITY_ESTIMATOR == "SG":
        vel_label = f"Savitzky-Golay window={SAVGOL_WINDOW}, order={SAVGOL_ORDER}"
    else:
        vel_label = f"Rolling window K={RW_WINDOW}"
    plt.suptitle(f"Velocity estimates ({vel_label}, m={M_SHAFT_KG} kg)",
                 fontsize=12, y=0.98)
    plt.tight_layout()
    return fig

def plot_Fsensed_over_time(friction_trials, force_levels):
    """Plot sensed outputted force over time for ecah force level (one subplot per level)."""
    fig, axes = plt.subplots(len(force_levels), 1,
                             figsize=(12, 4 * len(force_levels)), sharex=False)
    if len(force_levels) == 1:
        axes = [axes]

    colors = plt.cm.tab10(np.linspace(0,1,10))

    for i, force_mN in enumerate(force_levels): 
        ax = axes[i]
        recs = [r for r in friction_trials if r["force_label_mN"] == force_mN]
        for j, r in enumerate(recs):
            ax.plot(r["t_stream_s"], r["force_mN"],
                    linewidth=0.8, color=colors[j % len(colors)],
                    label=f"Trial {r['trial_num']}")
        ax.set_ylabel("Sensed force (mN)")
        ax.set_title(f"F_commanded = {force_mN} mN")
        ax.legend(fontsize=7, loc="upper right")
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time (s)")
    plt.suptitle(f"Sensed force output; filter strength = ___ m={M_SHAFT_KG} kg)", fontsize=12, y=0.98)
    plt.tight_layout()
    return fig

def plot_accel_over_time(friction_trials, force_levels):
    """Plot measured acceleration over time for each force level (one subplot per level)."""
    fig, axes = plt.subplots(len(force_levels), 1,
                             figsize=(12, 4 * len(force_levels)), sharex=False)
    if len(force_levels) == 1:
        axes = [axes]

    colors = plt.cm.tab10(np.linspace(0,1,10))

    for i, force_mN in enumerate(force_levels): 
        ax = axes[i]
        recs = [r for r in friction_trials if r["force_label_mN"] == force_mN]
        for j, r in enumerate(recs):
            ax.plot(r["t_stream_s"], r["accel_mmpss"],
                    linewidth=0.8, color=colors[j % len(colors)],
                    label=f"Trial {r['trial_num']}")
        ax.set_ylabel("Acceleration (mm/s^2)")
        ax.set_title(f"F_commanded = {force_mN} mN")
        ax.legend(fontsize=7, loc="upper right")
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time (s)")
    plt.suptitle(f"Measured acceleration; filter strength = ___ m={M_SHAFT_KG} kg)", fontsize=12, y=0.98)
    plt.tight_layout()
    return fig

def plot_stribeck_curve(friction_trials, force_levels):
    """Plot Stribeck curve: μ_d vs shaft speed v."""
    fig, ax = plt.subplots(figsize=(10, 6))

    speeds_all = []
    mu_d_all = []
    force_all = []

    for force_mN in force_levels:
        recs = [r for r in friction_trials if r["force_label_mN"] == force_mN]
        for r in recs:
            mask = r["t_stream_s"] >= T_IGNORE_S
            speed = r["velocity_mm_s"][mask]
            mu_d = r["mu_d"][mask]
            force = r["force_mN"][mask]
            speeds_all.extend(speed.tolist())
            mu_d_all.extend(mu_d.tolist())
            force_all.extend(force.tolist())

    sc = ax.scatter(speeds_all, mu_d_all, s=8, alpha=0.4, c=force_all,
                    cmap="viridis")
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("Sensed force (mN)")

    # Static friction reference lines
    ax.axhline(F_STATIC_LOW_MN / WEIGHT_MN, color="r", linestyle="--", alpha=0.5,
               label=f"μ_s range: {F_STATIC_LOW_MN/WEIGHT_MN:.4f}–{F_STATIC_HIGH_MN/WEIGHT_MN:.4f}")
    ax.axhline(F_STATIC_HIGH_MN / WEIGHT_MN, color="r", linestyle="--", alpha=0.5)

    ax.set_xlabel("Velocity (mm/s)")
    ax.set_xlim(np.percentile(speeds_all, 2), np.percentile(speeds_all, 98))
    ax.set_ylabel("μ_d = F_friction / (m·g)")
    ax.set_title(f"Stribeck curve (m={M_SHAFT_KG} kg, t ≥ {T_IGNORE_S} s)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

def plot_position_vs_mud_curve(friction_trials, force_levels):
    """Plot position vs mu_d"""
    fig, ax = plt.subplots(figsize=(10, 6))

    speeds_all = []
    mu_d_all = []
    force_all = []

    for force_mN in force_levels:
        recs = [r for r in friction_trials if r["force_label_mN"] == force_mN]
        for r in recs:
            mask = r["t_stream_s"] >= T_IGNORE_S
            speed = r["position_um"][mask]
            mu_d = r["mu_d"][mask]
            force = r["force_mN"][mask]
            speeds_all.extend(speed.tolist())
            mu_d_all.extend(mu_d.tolist())
            force_all.extend(force.tolist())

    sc = ax.scatter(speeds_all, mu_d_all, s=8, alpha=0.4, c=force_all,
                    cmap="viridis")
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label("Sensed force (mN)")

    # Static friction reference lines
    ax.axhline(F_STATIC_LOW_MN / WEIGHT_MN, color="r", linestyle="--", alpha=0.5,
               label=f"μ_s range: {F_STATIC_LOW_MN/WEIGHT_MN:.4f}–{F_STATIC_HIGH_MN/WEIGHT_MN:.4f}")
    ax.axhline(F_STATIC_HIGH_MN / WEIGHT_MN, color="r", linestyle="--", alpha=0.5)

    ax.set_xlabel("Shaft position (um)")
    ax.set_ylabel("μ_d = F_friction / (m·g)")
    ax.set_title(f"Faux positional Stribeck curve (m={M_SHAFT_KG} kg, t ≥ {T_IGNORE_S} s)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

def plot_stribeck_3d(friction_trials, force_levels):
    """Plot 3D Stribeck surface: position vs velocity vs μ_d, colored by sensed force."""
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection="3d")

    positions_all = []
    speeds_all = []
    mu_d_all = []
    force_all = []

    for force_mN in force_levels:
        recs = [r for r in friction_trials if r["force_label_mN"] == force_mN]
        for r in recs:
            mask = r["t_stream_s"] >= T_IGNORE_S
            positions_all.extend(r["position_um"][mask].tolist())
            speeds_all.extend(r["velocity_mm_s"][mask].tolist())
            mu_d_all.extend(r["mu_d"][mask].tolist())
            force_all.extend(r["force_mN"][mask].tolist())

    sc = ax.scatter(positions_all, speeds_all, mu_d_all,
                    s=6, alpha=0.4, c=force_all, cmap="viridis")
    cbar = fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.1)
    cbar.set_label("Sensed force (mN)")

    ax.set_xlabel("Position (µm)")
    ax.set_ylabel("Velocity (mm/s)")
    ax.set_zlabel("μ_d = F_friction / (m·g)")
    ax.set_title(f"3D Stribeck (m={M_SHAFT_KG} kg, t ≥ {T_IGNORE_S} s, shift={STREAM_FORCE_SHIFT})")

    plt.tight_layout()
    return fig